In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

labels = pd.read_csv("../../Human_Activity_Recognition/dataset/RawData/labels.txt", 
                    sep=r'\s+', header=None, 
                    names=['experiment_number_ID', 'user_number_ID', 'activity_number_ID', 'label_start_point', 'label_end_point'])

activity_labels = pd.read_csv("../../Human_Activity_Recognition/dataset/activity_labels.txt", 
                              sep=r'\s+', header=None, names=['activity_number_id', 'activity_label'])

data_dir = '../../Human_Activity_Recognition/dataset/RawData'

WINDOW_SIZE = 128 #samples per window, 2.56 s at 50 Hz
STEP = 128 #how far we slide between windows, each window starts exactly where the last ended, no overlap


In [ ]:
def extract_features(window):
    """Take one 128x6 window, return a flat row of 24 uniquely-named features."""
    feats = {}
    for col in window.columns:  # loop over the 6 signals
        feats[f'{col}_mean'] = window[col].mean()
        feats[f'{col}_std']  = window[col].std()
        feats[f'{col}_min']  = window[col].min()
        feats[f'{col}_max']  = window[col].max()
    return pd.Series(feats)

In [ ]:
def process_experiment(exp_num, user_num, labels, data_dir):
    """Load one experiment, window it, extract features.
    Returns three aligned lists: feature rows, activity labels, user ids."""

    # 1. Build the two filenames (with zero-padding)
    acc_path  = f'{data_dir}/acc_exp{exp_num:02d}_user{user_num:02d}.txt'
    gyro_path = f'{data_dir}/gyro_exp{exp_num:02d}_user{user_num:02d}.txt'

    # 2. Load both sensors
    acc  = pd.read_csv(acc_path,  sep=r'\s+', header=None, names=['acc_x','acc_y','acc_z'])
    gyro = pd.read_csv(gyro_path, sep=r'\s+', header=None, names=['gyro_x','gyro_y','gyro_z'])

    # 3. Glue into one 6-column signal
    signals = pd.concat([acc, gyro], axis=1)

    # 4. Filter labels to THIS experiment
    exp_labels = labels[labels['experiment_number_ID'] == exp_num]

    # 5. Window each segment + extract features
    feature_rows = []
    label_list   = []
    user_list    = []

    for _, seg in exp_labels.iterrows():
        start    = seg['label_start_point']
        end      = seg['label_end_point']
        activity = seg['activity_number_ID']

        segment = signals.iloc[start:end+1]

        for start_idx in range(0, len(segment) - WINDOW_SIZE + 1, STEP):
            window = segment.iloc[start_idx : start_idx + WINDOW_SIZE]
            feature_rows.append(extract_features(window))
            label_list.append(activity)
            user_list.append(user_num)

    return feature_rows, label_list, user_list

In [ ]:
f, l, u = process_experiment(1, 1, labels, data_dir)

print('windows:', len(f))
print('labels: ', len(l))
print('users:  ', len(u))
print('first label:', l[0], ' first user:', u[0])

In [ ]:
exp_user_pairs = labels[['experiment_number_ID', 'user_number_ID']].drop_duplicates()
print(exp_user_pairs.shape)
print(exp_user_pairs)

In [ ]:
all_features = []
all_activities = []
all_users = []

for _, pair in exp_user_pairs.iterrows():
    exp_num  = pair['experiment_number_ID']
    user_num = pair['user_number_ID']

    f, l, u = process_experiment(exp_num, user_num, labels, data_dir)

    all_features.extend(f)
    all_activities.extend(l)
    all_users.extend(u)

print('total windows:', len(all_features))
print('total labels: ', len(all_activities))
print('total users:  ', len(all_users))

In [ ]:
# 1. The feature matrix: 5773 rows x 24 feature columns
X = pd.DataFrame(all_features)

# 2. The labels, as a Series
y = pd.Series(all_activities, name='activity')

# 3. The user ids, as a Series (for subject-independent splitting later)
groups = pd.Series(all_users, name='user')

print('X shape:', X.shape)
print('y shape:', y.shape)
print('groups shape:', groups.shape)
X.head()

In [ ]:
y.value_counts()

In [ ]:
# filtering out the transition activities
# a True/False for every row: True if activity is 1-6
mask = y <= 6

# apply the SAME mask to all three, keeping only True rows
X_core = X[mask]
y_core = y[mask]
groups_core = groups[mask]

print('before:', X.shape[0], 'windows')
print('after: ', X_core.shape[0], 'windows')
print(y_core.value_counts())

In [ ]:
import os
os.makedirs('../data', exist_ok=True)

X_core.to_csv('../data/har_features.csv', index=False)
y_core.to_frame().to_csv('../data/har_labels.csv', index=False)
groups_core.to_frame().to_csv('../data/har_groups.csv', index=False)
print('saved to ../data/')